In [1]:
import numpy as np
import pickle
import optuna

In [2]:
# Load bounds
with open(r"D:\UST Project\UST_Analog_automation\notebooks\new_data_exp\trained_models\param_bounds.pkl", "rb") as f:
    bounds = pickle.load(f)

feature_names = list(bounds.keys())  # ['a','b','c','d']

# Log sampling config (optimizer-side only)
feature_config = {
    "a": {"log": True},
    "b": {"log": False},  # includes 0
    "c": {"log": True},
    "d": {"log": True},
}

# Load trained forward models
model_gain = pickle.load(open(r"D:\UST Project\UST_Analog_automation\notebooks\new_data_exp\trained_models\model_gain_xgboost.pkl", "rb"))
model_pm   = pickle.load(open(r"D:\UST Project\UST_Analog_automation\notebooks\new_data_exp\trained_models\model_pm_xgboost.pkl", "rb"))
model_ugf  = pickle.load(open(r"D:\UST Project\UST_Analog_automation\notebooks\new_data_exp\trained_models\model_ugf_xgboost.pkl", "rb"))


In [3]:
GAIN_TARGET = 18.742711		   # example
PM_MAX      = 87.258927   # constraint
UGF_TARGET  = 12.859450		  # example

In [4]:
# ============================================================
# 4. INVERSE LOSS FUNCTION
# ============================================================
def inverse_loss(x):
    """
    x shape: (1, 4)
    """
    gain = model_gain.predict(x)[0]
    pm   = model_pm.predict(x)[0]
    ugf  = model_ugf.predict(x)[0]

    loss = (
        abs(gain - GAIN_TARGET) +
        max(0.0, PM_MAX - pm) +
        abs(ugf - UGF_TARGET)
    )
    return loss


In [5]:
def objective(trial):
    x = []

    for f in feature_names:
        low, high = bounds[f]

        if feature_config[f]["log"]:
            val = trial.suggest_float(f, low, high, log=True)
        else:
            val = trial.suggest_float(f, low, high)

        x.append(val)

    x = np.array(x).reshape(1, -1)
    return inverse_loss(x)

In [6]:
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=50
    )
)

study.optimize(objective, n_trials=200, show_progress_bar=True)

[I 2026-03-04 17:30:56,441] A new study created in memory with name: no-name-148a3076-7a01-44d8-b30d-82eb04ecde25


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-03-04 17:30:56,463] Trial 0 finished with value: 13.036462783813477 and parameters: {'a': 6.59269988761102e-06, 'b': 0.0003698270095505816, 'c': 4.462395092148129e-06, 'd': 6.37710619087497e-05}. Best is trial 0 with value: 13.036462783813477.
[I 2026-03-04 17:30:56,468] Trial 1 finished with value: 26.337921142578125 and parameters: {'a': 5.478384061253937e-06, 'b': 6.068172801571452e-05, 'c': 2.521068595986405e-06, 'd': 7.999520174016835e-05}. Best is trial 0 with value: 13.036462783813477.
[I 2026-03-04 17:30:56,473] Trial 2 finished with value: 32.57509231567383 and parameters: {'a': 7.987991737671789e-06, 'b': 0.0002754395954973417, 'c': 2.44222604110033e-06, 'd': 8.734449411161188e-05}. Best is trial 0 with value: 13.036462783813477.
[I 2026-03-04 17:30:56,477] Trial 3 finished with value: 1.9678878784179688 and parameters: {'a': 9.717639958417919e-06, 'b': 8.259972294864981e-05, 'c': 2.7997458439219924e-06, 'd': 4.485592596776692e-05}. Best is trial 3 with value: 1.96788

In [7]:
best_params = study.best_params
x_best = np.array([best_params[f] for f in feature_names]).reshape(1, -1)

gain_best = model_gain.predict(x_best)[0]
pm_best   = model_pm.predict(x_best)[0]
ugf_best  = model_ugf.predict(x_best)[0]

print("\n===== INVERSE PREDICTION =====")
for k in feature_names:
    print(f"{k} = {best_params[k]:.6e}")

print("\nPredicted outputs:")
print(f"GAIN = {gain_best:.6f}")
print(f"PM   = {pm_best:.6f}")
print(f"UGF  = {ugf_best:.6f}")
print(f"Final loss = {study.best_value:.6f}")


===== INVERSE PREDICTION =====
a = 6.827679e-06
b = 8.316329e-05
c = 3.215640e-06
d = 4.593693e-05

Predicted outputs:
GAIN = 18.822449
PM   = 93.338928
UGF  = 13.144586
Final loss = 0.364874


In [8]:
import pandas as pd

df = pd.read_csv("Opam.csv")
pd.set_option('display.float_format', '{:.10f}'.format)


df.head()

,a,b,c,d,gain,pm,ugf
0,0.0000112000,0.0001320120,0.0000024000,0.0000384000,18.7427113100,87.2589267500,12.8594501900
1,0.0000112000,0.0001320120,0.0000024000,0.0000427000,20.0224857400,88.4842564900,14.2978920500
2,0.0000112000,0.0001320120,0.0000024000,0.0000469000,21.2202001200,89.6240725500,15.5129365100
3,0.0000112000,0.0001320120,0.0000024000,0.0000512000,22.3550526600,90.9081618400,17.4676334600
4,0.0000112000,0.0001320120,0.0000024000,0.0000555000,23.4437963200,92.1196945900,19.5720128900


In [9]:
print("\n===== STABILITY CHECK =====")
for i in range(5):
    noise = np.random.normal(0, 5e-8, size=x_best.shape)
    g = model_gain.predict(x_best + noise)[0]
    p = model_pm.predict(x_best + noise)[0]
    u = model_ugf.predict(x_best + noise)[0]
    print(f"{i+1}: GAIN={g:.4f}, PM={p:.4f}, UGF={u:.4f}")


===== STABILITY CHECK =====
1: GAIN=18.8224, PM=93.3389, UGF=13.1446
2: GAIN=18.8224, PM=93.3389, UGF=13.1446
3: GAIN=18.8224, PM=93.3389, UGF=13.1446
4: GAIN=18.8224, PM=93.3389, UGF=13.1446
5: GAIN=18.8224, PM=93.3389, UGF=13.1446


In [10]:
print("\n===== TOP 5 INVERSE SOLUTIONS =====")
top_trials = sorted(study.trials, key=lambda t: t.value)[:5]

for i, t in enumerate(top_trials):
    print(f"\nSolution {i+1} | Loss = {t.value:.6f}")
    for f in feature_names:
        print(f"{f} = {t.params[f]:.6e}")


===== TOP 5 INVERSE SOLUTIONS =====

Solution 1 | Loss = 0.364874
a = 6.827679e-06
b = 8.316329e-05
c = 3.215640e-06
d = 4.593693e-05

Solution 2 | Loss = 0.364874
a = 6.585031e-06
b = 5.687620e-05
c = 3.439008e-06
d = 4.410141e-05

Solution 3 | Loss = 0.364874
a = 6.540730e-06
b = 4.621925e-05
c = 3.436176e-06
d = 4.484351e-05

Solution 4 | Loss = 0.364874
a = 6.751392e-06
b = 4.482670e-05
c = 3.315431e-06
d = 4.616248e-05

Solution 5 | Loss = 0.364874
a = 6.444005e-06
b = 7.278521e-05
c = 3.217114e-06
d = 4.461746e-05


In [11]:
import pandas as pd 

df = pd.read_csv("Opam.csv")

df.head()

,a,b,c,d,gain,pm,ugf
0,0.0000112000,0.0001320120,0.0000024000,0.0000384000,18.7427113100,87.2589267500,12.8594501900
1,0.0000112000,0.0001320120,0.0000024000,0.0000427000,20.0224857400,88.4842564900,14.2978920500
2,0.0000112000,0.0001320120,0.0000024000,0.0000469000,21.2202001200,89.6240725500,15.5129365100
3,0.0000112000,0.0001320120,0.0000024000,0.0000512000,22.3550526600,90.9081618400,17.4676334600
4,0.0000112000,0.0001320120,0.0000024000,0.0000555000,23.4437963200,92.1196945900,19.5720128900
